# Discourse Lab — reference notebook

A guided tour of the toolbox, in build order (see `TODO.txt` and
`discourse-lab-dev.md` §6). Each section is runnable on its own once the
previous one has executed — copy a section into your own notebook as a
starting point.

Dynamics are numeric; language (the last section) is an offline pass over
already-decided numeric state and never runs inside the simulation loop.


## Running in Google Colab

Open this notebook in Colab (GitHub tab -> `lrodeck/Network-Simulation`,
branch `claude/todos-continuation-tupw4f` -> `notebooks/demo.ipynb`, or File ->
Upload notebook), then run the cell below first — it installs the package
straight from this repo (not on PyPI, and not yet merged to `main`, hence the
pinned branch) and is a no-op outside Colab. Everything after it is
unchanged: `workspace()` already resolves to `/content/dlab` under Colab
automatically.


In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "git+https://github.com/lrodeck/Network-Simulation.git@claude/todos-continuation-tupw4f"


In [2]:
import os
import warnings
import dataclasses

import numpy as np
import polars as pl

from discourse_lab.config import Config
from discourse_lab.io.workspace import figures_dir, tables_dir

# A scratch workspace so this notebook never touches a real ~/dlab.
os.environ["DLAB_HOME"] = os.path.join(os.getcwd(), "dlab_demo")

# The demo population/tick counts below are deliberately tiny for speed, which
# makes the cascade size/depth caps bind far more often than they would at a
# realistic scale — expected here, not a sign of a bug. Silence the warning
# for this notebook; leave it on when calibrating a real run.
warnings.filterwarnings("ignore", message="cascade:")

SEED = 0
rng = np.random.default_rng(SEED)


## 1. Config

Every run is a pure function of `(Config, seed)`. Nested, frozen, and
structurally hashed — the hash is what artifact caching and sweep
resumability key off of. This notebook uses a small population and a short
run so every cell finishes in seconds; swap in the defaults (`Config()`) for
anything you intend to actually analyze.


In [3]:
cfg = Config()
cfg = dataclasses.replace(
    cfg,
    population=dataclasses.replace(cfg.population, n_users=600, n_topics=4),
    # drift defaults to "full"; turned off here so sections 1-8 isolate the
    # dynamics this notebook is actually demonstrating. Turned back on in §9.
    dynamics=dataclasses.replace(cfg.dynamics, n_ticks=30, kernel="homophily", ranker="affinity", drift="none"),
)
print("config hash:", cfg.hash())
print("stance dims:", cfg.stance_dims())
cfg


config hash: e38e9ceec30bc3bea79ad23af2255c31
stance dims: 3


Config(population=PopulationConfig(n_users=600, n_topics=4, stance_dims=3, archetype_weights=(), archetype_offsets=(), correlation_pairs=(), activity_sigma=1.8, pareto_alpha=2.3, topic_logit_sigma=1.0), graph=GraphConfig(generator='latent_space', mean_degree=40.0, homophily_beta=0.35, prominence_gamma=0.6, mirror_p=0.02, fanout_cap=400, knn_k=60, long_tie_fraction=0.1, pa_fraction=0.35, sbm_blocks=0, sbm_homophily=0.8), dynamics=DynamicsConfig(n_ticks=30, posts_per_tick_rate=0.02, ticks_per_day=24, fatigue_decay=0.9, attention_budget=30.0, tau_position=6.0, inject_k=0, ranker='affinity', kernel='homophily', kernel_theta=(), agreement_metric='rms', hawkes_mu0=0.004, hawkes_ratio=0.6, hawkes_beta=1.5, max_thread_age=15, hawkes_mu_inherit=1.0, max_replies_per_tick=1, trend_eta=0.3, post_lifetime=5, rho_s=0.9, rho_sigma=0.9, cascade_depth_decay=0.7, max_cascade_depth=25, max_cascade_size=1000, drift='none', drift_lr=0.02, drift_lr_social=0.01, drift_ramp_ticks=50, ou_k=(), noise_sigma=0.00

## 2. Population

Archetype mixture over a correlated Gaussian latent, transformed to target
marginals by a Gaussian copula (spec §2.1). `cached_population` reuses a
prior draw for this exact population sub-config + seed.


In [4]:
from discourse_lab.population import cached_population

pop = cached_population(cfg, seed=SEED, rng=rng)
print(f"{len(pop.trait_names)} traits x {cfg.population.n_users} users")
print("archetypes:", sorted(set(pop.archetype_names)))

archetype_of_user = np.array(pop.archetype_names)[pop.archetype_labels]
pl.DataFrame(
    {
        "archetype": archetype_of_user,
        "activity": pop.X_used[:, pop.trait_names.index("activity")],
        "prominence": pop.X_used[:, pop.trait_names.index("prominence")],
        "contrarianism": pop.X_used[:, pop.trait_names.index("contrarianism")],
    }
).group_by("archetype").agg(pl.all().mean()).sort("archetype")


27 traits x 600 users
archetypes: [np.str_('firebrand'), np.str_('institution'), np.str_('lurker'), np.str_('newcomer'), np.str_('poster')]


archetype,activity,prominence,contrarianism
str,f64,f64,f64
"""firebrand""",3.448825,1.628958,0.504463
"""institution""",7.109182,12.516846,0.35226
"""lurker""",0.634113,1.561538,0.262807
"""newcomer""",3.330958,1.801692,0.248533
"""poster""",14.455528,1.506326,0.291944


## 3. Graph

`latent_space` (the default) connects users by homophily in stance/topic
space plus a preferential-attachment term on prominence, calibrated by
bisection to hit `mean_degree`. Swappable for `sbm`, `configuration_model`,
or `barabasi_albert` via `cfg.graph.generator` — each is a useful null model
for isolating what homophily itself contributes.


In [5]:
from discourse_lab.network import cached_graph
from discourse_lab.network.measures import degree_sequence, global_clustering

graph = cached_graph(cfg, seed=SEED, pop=pop, rng=rng)
deg = degree_sequence(graph.csr)
print(f"mean degree: {deg.mean():.1f} (target {cfg.graph.mean_degree})")
print(f"clustering coefficient: {global_clustering(graph.csr):.3f}")


mean degree: 40.1 (target 40.0)
clustering coefficient: 0.280


## 4. Stance editor widget

Draw a population's stance distribution by hand; the sampler preview shows
exactly what a copula draw from that curve looks like. Autosaves to
`scenarios/<name>.json` in the same schema `ScenarioConfig.from_editor_json`
reads, so a drawn scenario plugs directly back into a `Config`.


In [6]:
from discourse_lab.widgets import StanceEditorWidget

stance_editor = StanceEditorWidget(name="demo")
stance_editor


## 5. Running the simulation

`run_iter(cfg, seed)` is the generator core — timing, generation, exposure,
reaction, cascades, perception, and the discourse-state update, one tick at a
time (drift is off in this `cfg`; see §9). `cached_run`/`run` collect it
into a persisted, parquet-backed run directory.


In [7]:
from discourse_lab.runner import cached_run, load_run

# persist=... opts into the raw tables. Without "posts" the §5.1 facts below
# cannot be computed at all: per-post engagement, cascade roots and depths all
# live there, not in the per-tick metrics.
run_dir = cached_run(cfg, seed=SEED, persist=("posts", "engagements"))
handle = load_run(cfg, seed=SEED)
metrics = handle.metrics()
print("run dir:", os.path.relpath(run_dir))
metrics.select(["t", "n_posts", "n_replies", "n_exposures", "attention_gini", "r_eff"]).tail(5)

run dir: dlab_demo/runs/e38e9ceec30bc3bea79ad23af2255c31/0


t,n_posts,n_replies,n_exposures,attention_gini,r_eff
i64,f64,f64,f64,f64,f64
25,38.0,3.0,2915.0,0.602843,0.029503
26,28.0,4.0,2973.0,0.600455,0.048436
27,35.0,3.0,2973.0,0.584652,0.070299
28,36.0,4.0,2885.0,0.601114,0.05026
29,36.0,1.0,2870.0,0.603508,0.042509


In [8]:
# Two different attention Ginis, and they do not agree.
#
#   metrics.parquet:attention_gini  rolling, over posts still active this tick
#   posts.parquet                   lifetime engagement total per post
#
# spec §5.1's 0.8-0.95 target is about the second. Reporting the first against
# that range compares different quantities, which is what this cell used to do.
from discourse_lab.measures import gini

rolling = float(metrics["attention_gini"].drop_nulls().mean())
lifetime = float(gini(handle.posts()["engagement_count"].to_numpy().astype(float)))
print(f"rolling (per tick, active posts): {rolling:.3f}")
print(f"lifetime (per post, spec §5.1):   {lifetime:.3f}")

rolling (per tick, active posts): 0.584
lifetime (per post, spec §5.1):   0.517


## 6. Run monitor widget (live)

Consumes `run_iter` directly and plots measures as ticks complete — the
salience/agreement pair, bubble index, attention Gini, R_eff — so a config's
fate is visible within twenty ticks rather than only at the end.


In [9]:
from discourse_lab.runner import run_iter
from discourse_lab.widgets import RunMonitorWidget

monitor = RunMonitorWidget()
for state in run_iter(cfg, seed=SEED + 1):  # a fresh seed; population/graph artifacts are reused
    monitor.push(state)
monitor


## 7. Post-run metrics — the spec §5.1 calibration table

Computed from the run's own persisted posts and engagements, not from
synthetic data.

**This demo runs a deliberately tiny population**, and several of these facts
are scale-dependent — clustering against a degree-matched null measures ~1.8
here but 3.3 at 2000 users and 5.1 at 3000. Read the failures below as "this
demo is small", not as "the model misses them". The table also marks rows it
will not grade: a fact the spec quotes no target for, and the engagement
exponent whenever the tail is not a power law, where the fitted value depends
on where `x_min` lands rather than on the model.

In [10]:
from discourse_lab.metrics import stylized_facts_from_run
from discourse_lab.viz import tables

report = stylized_facts_from_run(handle, graph=graph, pop=pop)
facts = tables.stylized_facts_table(report)
print(tables.to_markdown(facts))

| Fact | Value | Target | Status |
| --- | --- | --- | --- |
| Engagement per post (power-law alpha) — not a power law | 2.042 | 2-3 | n/a |
| Cascade size (share of singletons) | 0.867 | 0.9-1 | FAIL |
| Thread depth (mean, branched cascades) | 1.159 | 1.5-3 | FAIL |
| Attention Gini (lifetime, per post) | 0.517 | 0.8-0.95 | FAIL |
| Posting volume Gini | 0.715 | 0.7-0.9 | pass |
| Reciprocity | 0.378 | 0.2-0.4 | pass |
| Clustering vs degree-matched null | 1.832 | >= 3 | FAIL |
| Inter-cluster interaction rate | 0.242 | 0-0.33 | pass |
| Hostility given cross-cluster contact | 0.019 | -- | n/a |


## 8. Experiment 1 — kernel/ranker sweep vs. null

Every effect is measured against a matched `kernel="null"` run (same
population, graph, activity). Kept tiny here (2 kernels x 1 ranker x 3
seeds); bump `seeds` to 10+ for anything you'd actually report.


In [11]:
from discourse_lab.experiments import build_experiment1, run_experiment1, summarize_experiment1

# spec §4.4: "Minimum 10 seeds, and report distributions rather than points.
# This is the single most common way simulation studies of this kind go wrong."
# Three seeds here only to keep the demo quick — do not read an effect off it.
cells_exp1 = build_experiment1(cfg, kernels=("homophily", "bandwagon"), rankers=("affinity",))
rows = run_experiment1(cells_exp1, seeds=[0, 1, 2])
summarize_experiment1(rows)

kernel,ranker,attention_gini_effect_mean,bubble_index_effect_mean,r_eff_effect_mean,agreement_effect_mean,salience_effect_mean,attention_gini_effect_std,bubble_index_effect_std,r_eff_effect_std,agreement_effect_std,salience_effect_std
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""homophily""","""affinity""",-0.003681,-0.00194,0.001489,0.010916,0.006151,0.010249,0.002467,0.008763,0.00633,0.007436
"""bandwagon""","""affinity""",0.153551,-0.003253,0.370797,0.03699,0.058371,0.016873,0.002227,0.043833,0.011471,0.008221


## 9. Drift (optional)

Two free channels (reinforcement on expression traits, social influence on
stance) plus Ornstein-Uhlenbeck mean-reversion. Off by default
(`dynamics.drift`); gains ramp linearly from 0 over `drift_ramp_ticks` when
it's on, so switching it on mid-analysis doesn't jolt the population.


In [12]:
cfg_drift = dataclasses.replace(
    cfg,
    dynamics=dataclasses.replace(cfg.dynamics, drift="full", drift_ramp_ticks=10, n_ticks=20),
)
drift_states = list(run_iter(cfg_drift, seed=SEED + 2))
drift_traj = pl.DataFrame([{"t": s.t, **s.metrics} for s in drift_states])
drift_traj.select(["t", "n_posts", "attention_gini", "salience", "agreement"]).tail(5)

t,n_posts,attention_gini,salience,agreement
i64,f64,f64,f64,f64
15,11.0,0.550942,0.637602,-1.192363
16,14.0,0.501749,0.627803,-1.218372
17,21.0,0.593152,0.622685,-1.191229
18,23.0,0.621922,0.632889,-1.194614
19,34.0,0.591253,0.625373,-1.123941


## 10. LLM realization (optional, offline pass)

Never inside the tick. Quantization and prompt-building need no network and
always run below; the actual call to
[Ollama Cloud](https://ollama.com) only fires if `OLLAMA_API_KEY` is set.

```bash
export OLLAMA_API_KEY=...   # https://ollama.com/settings/keys
```


In [13]:
from discourse_lab.dynamics import ExpressionMap, generate_posts
from discourse_lab.llm.voice_card import build_voice_card_messages, fit_bands, user_bands

K, D = cfg.population.n_topics, cfg.stance_dims()
expr = ExpressionMap.build(pop.trait_names, K)
authors = rng.choice(cfg.population.n_users, size=5, replace=True)
demo_posts = generate_posts(authors, pop, expr, np.zeros(K), np.zeros((K, D)), eta=0.3, rng=rng)

bands = fit_bands(pop)
example_bands = user_bands(pop, bands, int(authors[0]))
print("quantized traits (never raw floats in the prompt):", example_bands)

messages = build_voice_card_messages(pop.archetype_names[pop.archetype_labels[authors[0]]], example_bands)
print()
print(messages[1]["content"])


quantized traits (never raw floats in the prompt): {'openness': 'very high', 'conscientiousness': 'low', 'extraversion': 'very low', 'agreeableness': 'medium', 'neuroticism': 'very low', 'plasticity': 'very high', 'conviction': 'high', 'contrarianism': 'very low', 'credulity': 'high'}

Archetype: poster

Trait profile:
- openness: very high
- conscientiousness: low
- extraversion: very low
- agreeableness: medium
- neuroticism: very low
- plasticity: very high
- conviction: high
- contrarianism: very low
- credulity: high

Return exactly this format, nothing else:
PERSONA: <three sentences describing who this person is and how they post>
TICS: <tic one>; <tic two>; <tic three>
REGISTER: <one line on formality/vocabulary/punctuation habits>


In [14]:
if os.environ.get("OLLAMA_API_KEY"):
    from discourse_lab.llm import OllamaCloudClient, realize

    client = OllamaCloudClient(model="gpt-oss:120b-cloud")
    texts = realize(client, cfg, demo_posts, pop, post_ids=[int(demo_posts.id[0])])
    print(texts)
else:
    print("OLLAMA_API_KEY not set — skipping the live call. The prompt above is what would be sent.")


OLLAMA_API_KEY not set — skipping the live call. The prompt above is what would be sent.


## 11. Figures and tables

Everything above, written to `dlab/figures/` and `dlab/tables/` as vector PDF
(for a paper) and PNG (for here), with each table in both LaTeX booktabs and
Markdown. Requires the optional extra: `pip install -e ".[viz]"`.

`fig_metric_trajectories` takes the run **and its matched null** — spec §5.3
makes that comparison mandatory, so there is no way to plot the model alone.

In [15]:
from discourse_lab.viz import (
    fig_calibration, fig_cascade_sizes, fig_engagement_ccdf,
    fig_degree_ccdf, fig_lorenz, fig_metric_trajectories, save_figure,
)

# the matched null: same population, same graph, same seed — only the kernel
# differs, which is what makes the difference attributable to the kernel
cfg_null = dataclasses.replace(cfg, dynamics=dataclasses.replace(cfg.dynamics, kernel="null"))
cached_run(cfg_null, seed=SEED, persist=("posts",))
handle_null = load_run(cfg_null, seed=SEED)

posts = handle.posts()
engagement = posts["engagement_count"].to_numpy().astype(float)
post_authors = posts.filter(posts["kind"] == "post")["author"].to_numpy()
per_user = np.bincount(post_authors, minlength=cfg.population.n_users).astype(float)

figures = {
    "fig_calibration": fig_calibration(report),
    "fig_metric_trajectories": fig_metric_trajectories(handle, handle_null),
    "fig_engagement_ccdf": fig_engagement_ccdf(engagement),
    "fig_cascade_sizes": fig_cascade_sizes(posts["root"].to_numpy()),
    "fig_lorenz": fig_lorenz({"attention (per post)": engagement,
                              "posting volume (per user)": per_user}),
    "fig_degree_ccdf": fig_degree_ccdf(graph.csr),
}
for name, figure in figures.items():
    save_figure(figure, name)

tables.save_table(facts, "T1_stylized_facts",
                  caption="Stylized facts against spec 5.1 targets.", label="tab:facts")
tables.save_table(tables.experiment_effects_table(rows), "T2_experiment1_effects",
                  caption="Experiment 1 effects against the matched null.", label="tab:effects")
tables.save_table(tables.provenance_table(handle.meta), "T3_provenance",
                  caption="Run provenance.", label="tab:provenance")

print("figures ->", figures_dir())
print("tables  ->", tables_dir())
figures["fig_calibration"]

figures -> /home/user/Network-Simulation/notebooks/dlab_demo/figures
tables  -> /home/user/Network-Simulation/notebooks/dlab_demo/tables


<Figure size 700x478 with 1 Axes>